### Задание 1

Импортируйте файлы workouts, payments и users. Преобразуйте типы данных "дата+время"

In [1]:
import os
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
path = '/content/drive/MyDrive/урок 6'
users = pd.read_excel(os.path.join(path, "users.xlsx"))
workouts = pd.read_csv(os.path.join(path, "workouts.csv"))
payments = pd.read_excel(os.path.join(path, "payments.xlsx"))

users['first_contact_datetime'] = pd.to_datetime(users['first_contact_datetime'])
workouts['start_at'] = pd.to_datetime(workouts['start_at'])
payments['payment_date'] = pd.to_datetime(payments['payment_date'])

In [4]:
users.head()

,first_contact_datetime,age,workouts_successful,workouts_total,months_active,user_id,region,geo_group
0,2020-08-08 13:57:25,NaN,10,13,2,2790000,NaN,СНГ
1,2020-05-02 00:30:02,NaN,26,38,7,780106,NaN,СНГ
2,2019-06-27 13:10:33,30.0,27,33,6,1133376,NaN,СНГ
3,2020-04-22 15:37:58,22.0,59,59,8,1996499,NaN,СНГ
4,2016-06-23 16:21:40,24.0,8,12,2,57899,Москва и Московская область,Москва


In [5]:
workouts.head()

,workout_id,cost,start_at,status,workout_schedule_type,client_id,client_status,workout_type,trainer_department,trainer_id
0,30793909,NaN,2020-11-13 12:45:00,success,trial,22034,NaN,general,Sales,10722051
1,31123309,750.0,2020-11-21 13:00:00,success,regular,22034,new,general,Spartacus,940642
2,31412167,750.0,2020-11-28 13:00:00,success,regular,22034,new,general,Spartacus,940642
3,31703605,750.0,2020-12-05 13:00:00,success,regular,22034,active,general,Spartacus,940642
4,26904500,NaN,2020-08-06 18:20:00,success,trial,88101,NaN,general,Sales,3826530


In [6]:
payments.head()

,user_id,payment_id,workout_type,amount,payment_date
0,132815,1535249,general,10337.768848,2020-01-30 15:03:59
1,165987,2395447,general,9711.925350,2020-08-18 21:59:09
2,17364,2641443,general,8951.474487,2020-10-12 11:55:00
3,148617,2402771,general,9981.884731,2020-08-20 21:30:27
4,175413,2035719,general,8848.691858,2020-05-22 20:10:55


In [7]:
users.dtypes

,0
first_contact_datetime,datetime64[ns]
age,float64
workouts_successful,int64
workouts_total,int64
months_active,int64
user_id,int64
region,object
geo_group,object


In [8]:
workouts.dtypes

,0
workout_id,int64
cost,float64
start_at,datetime64[ns]
status,object
workout_schedule_type,object
client_id,int64
client_status,object
workout_type,object
trainer_department,object
trainer_id,int64


In [9]:
payments.dtypes

,0
user_id,int64
payment_id,int64
workout_type,object
amount,float64
payment_date,datetime64[ns]


### Задание 2

~~~ sql
select    workout_type
        , count(workout_id) as cnt
from workouts
group by workout_type
order by cnt desc
~~~

In [10]:
####
workouts_cnt = workouts.groupby('workout_type').count()[['workout_id']].reset_index().rename(columns={'workout_id': 'cnt'}).sort_values('cnt', ascending=False)
workouts_cnt

,workout_type,cnt
2,general,103492
0,cycling,1175
1,functional test,26


### Задание 3

~~~ sql
select    date_part('month', start_at) as dm
        , trainer_department
        , count(workout_id) as cnt
from workouts
group by   dm
         , trainer_department
order by dm
~~~

In [11]:
####
workouts['dm'] = workouts['start_at'].dt.month
workouts_month = workouts.groupby(['dm', 'trainer_department'])['workout_id'].count().reset_index().rename(columns={'workout_id': 'cnt'}).sort_values('dm', ascending=True)
workouts_month


,dm,trainer_department,cnt
0,1,Athletic,11
1,1,Consultant,164
2,1,Cycling,1
3,1,Dinamo,36
4,1,Sales,457
...,...,...,...
79,12,Cycling,77
80,12,Dinamo,254
81,12,Sales,22
82,12,Spartacus,2147


### Задание 4

~~~ sql
select    date_trunc('month', payment_date) as p_tm
        , sum(amount)/1000 as sum_amt
from payments
where workout_type = 'general'
group by p_tm
order by p_tm
~~~

In [21]:
####

# 1. Фильтруем строки (WHERE)
payments_general = payments[payments["workout_type"] == "general"].copy()

# 2. Округляем даты до месяца (DATE_TRUNC)
payments['p_tm'] = payments['payment_date'].dt.to_period('M')

# 3. Группируем и считаем сумму (GROUP BY + SUM)
result = payments_general.groupby(['p_tm'])["amount"].sum().reset_index(name="sum_amt")

# 4. Делим на 1000 (/1000)
result["sum_amt"] = result["sum_amt"] / 1000

# 5. Сортируем по дате (ORDER BY)
payments_month = result.sort_values(by="p_tm", ascending=True).reset_index(drop=True)

# Выводим результат
payments_month





,p_tm,sum_amt
0,2020-01,4370.772775
1,2020-02,6114.601296
2,2020-03,7949.117944
3,2020-04,11908.988560
4,2020-05,11125.842432
5,2020-06,9682.458175
6,2020-07,9553.436788
7,2020-08,8345.359401
8,2020-09,9876.673936
9,2020-10,11044.861465


### Задание 5

~~~ sql
select    user_id
        , min(date_trunc('day', payment_date)) as first_date
from payments p
    join users u
        on p.user_id = u.user_id
where region is not null
group by user_id
order by user_id
limit 20
~~~

In [22]:
####
payments['first_date'] = payments['payment_date'].dt.to_period('D')
new_df = payments.merge(users, on = 'user_id', how = 'inner')
first_date = (new_df[new_df['region'].notnull()].groupby(['user_id']).agg(first_date=('first_date', 'min')).\
reset_index().sort_values('user_id', ascending = True)).head(20)
first_date

,user_id,first_date
0,185,2020-04-02
1,1455,2020-02-03
2,1558,2020-05-12
3,5312,2020-02-21
4,5804,2020-01-15
5,6877,2020-11-03
6,7226,2020-03-29
7,7824,2020-02-22
8,8881,2020-04-05
9,8913,2020-08-01


### Задание 6

~~~ sql
select    avg(age) as avg_age
from users p
    left join payments u
        on p.user_id = u.user_id
where payment_date is null
~~~

In [23]:
joined_df = users.merge(payments, on='user_id', how='left')
new_df = joined_df[joined_df['payment_date'].isna()].agg(avg_age=('age','mean'))
new_df

,age
avg_age,29.5


### Задание 7

~~~ sql
select   p_tm
       , sum(case when workout_type<>'general' then 1.0 else 0.0 end) / count(payment_id) as share_not_general
from payments
group by p_tm
~~~

In [24]:
path = '/content/drive/MyDrive/урок 6'
users = pd.read_excel(os.path.join(path, "users.xlsx"))
workouts = pd.read_csv(os.path.join(path, "workouts.csv"))
payments = pd.read_excel(os.path.join(path, "payments.xlsx"))

users['first_contact_datetime'] = pd.to_datetime(users['first_contact_datetime'])
workouts['start_at'] = pd.to_datetime(workouts['start_at'])
payments['payment_date'] = pd.to_datetime(payments['payment_date'])


In [25]:
import pandas as pd
import numpy as np
payments['p_tm'] = payments['payment_date'].dt.to_period('M')
payments['flag_not_general'] = np.where(payments['workout_type'] != 'general',1.0,0.0)

result_df = payments.groupby('p_tm').agg(
    sum_flag_not_general= ('flag_not_general','sum'), cnt_payment = ('payment_id','count')).reset_index()
result_df['share_not_general'] = result_df['sum_flag_not_general']/result_df['cnt_payment']
share_not_general_payments = result_df[['p_tm','share_not_general']]
share_not_general_payments

,p_tm,share_not_general
0,2020-01,0.004255
1,2020-02,0.012085
2,2020-03,0.006969
3,2020-04,0.007009
4,2020-05,0.003317
5,2020-06,0.010436
6,2020-07,0.007656
7,2020-08,0.007642
8,2020-09,0.017479
9,2020-10,0.017843


### Задание 8

~~~ sql
select    w.clientId
        , cnt_workout
        , amt
from (select    client_id
              , count(workout_id) as cnt_workout
     from workouts
     group by client_id
     ) w
    join (select      user_id
                    , sum(amount) as amt
          from payments
          group by user_id
          ) p
        on p.user_id = w.client_id
limit 10
~~~

In [26]:
####
path = '/content/drive/MyDrive/урок 6'
users = pd.read_excel(os.path.join(path, "users.xlsx"))
workouts = pd.read_csv(os.path.join(path, "workouts.csv"))
payments = pd.read_excel(os.path.join(path, "payments.xlsx"))

users['first_contact_datetime'] = pd.to_datetime(users['first_contact_datetime'])
workouts['start_at'] = pd.to_datetime(workouts['start_at'])
payments['payment_date'] = pd.to_datetime(payments['payment_date'])

In [30]:
workouts.head()


,workout_id,cost,start_at,status,workout_schedule_type,client_id,client_status,workout_type,trainer_department,trainer_id
0,30793909,NaN,2020-11-13 12:45:00,success,trial,22034,NaN,general,Sales,10722051
1,31123309,750.0,2020-11-21 13:00:00,success,regular,22034,new,general,Spartacus,940642
2,31412167,750.0,2020-11-28 13:00:00,success,regular,22034,new,general,Spartacus,940642
3,31703605,750.0,2020-12-05 13:00:00,success,regular,22034,active,general,Spartacus,940642
4,26904500,NaN,2020-08-06 18:20:00,success,trial,88101,NaN,general,Sales,3826530


In [36]:
w = workouts.groupby('client_id')['workout_id'].count().reset_index(name='cnt_workout')
p= payments.groupby('user_id')['amount'].sum().reset_index(name='amt')
new_t = w.merge(p, left_on = ['client_id'], right_on = ['user_id'] )
total = new_t[['client_id', 'cnt_workout', 'amt']]
total.head()

,client_id,cnt_workout,amt
0,185,50,29033.865003
1,1455,64,149649.891518
2,1558,13,18259.280262
3,5312,16,28416.673028
4,5804,46,38121.522478


In [ ]:
################################################################################################################################
################################################################################################################################
################################################################################################################################